# 21 — Client Intelligence Monitor

Give it a company name. A ReAct agent searches four mock sources — news, regulatory filings, leadership changes, market signals — and produces a typed `ClientIntelBrief` with recommended account actions.

In [ ]:
!pip install -q langgraph langchain-openai langchain-core python-dotenv

In [ ]:
import os
os.environ['OPENAI_API_KEY'] = 'sk-...'  # replace

In [ ]:
from typing import Literal
from pydantic import BaseModel, Field

class FundingEvent(BaseModel):
    round_type: str = Field(description='Funding round type.')
    amount_usd_m: float = Field(description='Amount in USD millions.')
    date: str = Field(description='Approximate date.')
    lead_investor: str = Field(description='Lead investor name.')

class LeadershipChange(BaseModel):
    role: str = Field(description='Job title.')
    change_type: Literal['hire', 'departure', 'promotion'] = Field(description='Nature of change.')
    name: str = Field(description='Person name.')
    date: str = Field(description='Approximate date.')

class RegulatoryExposure(BaseModel):
    topic: str = Field(description='Regulatory area.')
    severity: Literal['low', 'medium', 'high'] = Field(description='Impact severity.')
    summary: str = Field(description='One-sentence description.')

class StrategicSignal(BaseModel):
    signal: str = Field(description='Strategic move description.')
    implication: str = Field(description='Why it matters.')

class ClientIntelBrief(BaseModel):
    company: str = Field(description='Company name.')
    funding_events: list[FundingEvent] = Field(description='Funding rounds.')
    leadership_changes: list[LeadershipChange] = Field(description='Leadership changes.')
    regulatory_exposures: list[RegulatoryExposure] = Field(description='Regulatory risks.')
    strategic_signals: list[StrategicSignal] = Field(description='Strategic signals.')
    relationship_actions: list[str] = Field(description='Recommended account actions.')

In [ ]:
from langchain_core.messages import SystemMessage
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langgraph.prebuilt import create_react_agent

_NEWS = {'acme corp': 'Acme Corp raised $120M Series C (Tiger Global, Q1 2024). Partnership with Microsoft Azure announced.'}
_FILINGS = {'acme corp': 'Under FTC antitrust scrutiny following 2023 acquisition of Streamline Inc.'}
_LEADERSHIP = {'acme corp': 'CFO Sarah Lin departed March 2024. David Park hired from Goldman Sachs, effective May 2024.'}
_MARKET = {'acme corp': 'Stated intent to expand APAC by end-2024; hiring aggressively in Singapore.'}

@tool
def search_news(company: str) -> str:
    """Recent press coverage."""
    return _NEWS.get(company.lower(), f'No news for {company}.')

@tool
def search_regulatory_filings(company: str) -> str:
    """Regulatory filings."""
    return _FILINGS.get(company.lower(), f'No filings for {company}.')

@tool
def search_leadership_changes(company: str) -> str:
    """Leadership changes."""
    return _LEADERSHIP.get(company.lower(), f'No changes for {company}.')

@tool
def search_market_signals(company: str) -> str:
    """Market signals."""
    return _MARKET.get(company.lower(), f'No signals for {company}.')

_SYS = SystemMessage('You are a client intelligence analyst. Use tools to research the company, then summarise findings.')

def run(company):
    llm = ChatOpenAI(model='gpt-4o-mini', temperature=0)
    agent = create_react_agent(llm, [search_news, search_regulatory_filings, search_leadership_changes, search_market_signals], prompt=_SYS)
    result = agent.invoke({'messages': [{'role': 'user', 'content': f'Build an intelligence brief for: {company}'}]})
    findings = result['messages'][-1].content
    return ChatOpenAI(model='gpt-4o-mini', temperature=0).with_structured_output(ClientIntelBrief).invoke(
        f'Convert these findings into a ClientIntelBrief.\n\nCompany: {company}\n\n{findings}'
    )

In [ ]:
brief = run('Acme Corp')
print(f'Company: {brief.company}')
if brief.funding_events:
    print('\nFunding:')
    for f in brief.funding_events:
        print(f'  {f.date} {f.round_type} ${f.amount_usd_m:.0f}M ({f.lead_investor})')
if brief.leadership_changes:
    print('\nLeadership:')
    for lc in brief.leadership_changes:
        print(f'  {lc.date} {lc.role} {lc.change_type} - {lc.name}')
if brief.regulatory_exposures:
    print('\nRegulatory:')
    for r in brief.regulatory_exposures:
        print(f'  [{r.severity.upper()}] {r.topic}: {r.summary}')
if brief.strategic_signals:
    print('\nSignals:')
    for s in brief.strategic_signals:
        print(f'  * {s.signal} -> {s.implication}')
print('\nActions:')
for i, a in enumerate(brief.relationship_actions, 1):
    print(f'  {i}. {a}')

## Exercise

Replace `search_news` with a real Exa or Tavily search tool. The schema and synthesis step stay the same — only the data source changes.